In [1]:
import concurrent.futures
import os
import pandas as pd
from tqdm import tqdm
import scanpy as sc
if not os.path.exists('../../../data/rna/pseudobulk/outputs'):
    os.makedirs('../../../data/rna/pseudobulk/outputs')

## BMMC

### 1. Create psuedo counts

In [2]:
%run ../../00-utilities/functions/python/aggregate_counts.py

In [3]:
adata = sc.read_h5ad(
    '../../../data/rna/final-objects/final-bmmc-raw.h5ad'
)

In [4]:
adata = adata[adata.obs['aifi_celltype_l1'] != 'plasma']

In [5]:
print('Miniumum value:', adata.X.min())
print('Maximum value:', adata.X.max())

Miniumum value: 0
Maximum value: 74012


In [6]:
groups = adata.obs["sample.sampleKitGuid"].unique().tolist()
adata_dict = {
    group: adata[adata.obs["sample.sampleKitGuid"] == group].copy() for group in groups
}

In [7]:
outputs_dir  = '../../../data/rna/pseudobulk/outputs'

In [8]:
with concurrent.futures.ProcessPoolExecutor(max_workers=5) as executor:
    tasks = [(sample_id, adata, 'aifi_plot_l3', outputs_dir, 'bmmc') for sample_id, adata in adata_dict.items()]
    list(tqdm(executor.map(process_sample_wrapper, tasks), total=len(tasks)))

100%|██████████| 69/69 [02:57<00:00,  2.57s/it]


### 1.2. Create psuedo counts with plasma

In [4]:
print('Miniumum value:', adata.X.min())
print('Maximum value:', adata.X.max())

Miniumum value: 0
Maximum value: 78954


In [5]:
groups = adata.obs["sample.sampleKitGuid"].unique().tolist()
adata_dict = {
    group: adata[adata.obs["sample.sampleKitGuid"] == group].copy() for group in groups
}

In [6]:
outputs_dir  = '../../../data/rna/pseudobulk/outputs'

In [7]:
with concurrent.futures.ProcessPoolExecutor(max_workers=5) as executor:
    tasks = [(sample_id, adata, 'aifi_plot_l3', outputs_dir, 'bmmc') for sample_id, adata in adata_dict.items()]
    list(tqdm(executor.map(process_sample_wrapper, tasks), total=len(tasks)))

100%|██████████| 69/69 [02:59<00:00,  2.61s/it]


### 2. Create a metadata file for DEseq2

In [9]:
# Compute per-sample × celltype cell counts, plus total cells per sample
cell_counts = (
    # Count cells per (sample, celltype)
    adata.obs.groupby(["sample.sampleKitGuid", "aifi_plot_l3"], observed=True)
      .size()
      .reset_index(name="n_cells")
      # Merge with total cells per sample
      .merge(
          adata.obs.groupby("sample.sampleKitGuid", observed=True)
            .size()
            .reset_index(name="total_cells"),
          on="sample.sampleKitGuid",
          how="left"
      )
)

# Add a flag column indicating whether a sample–celltype passes thresholds:
#   - at least 10 cells in that sample × celltype
#   - at least 1000 total cells in the sample overall
cell_counts["keep"] = (
    (cell_counts["n_cells"] >= 10) & (cell_counts["total_cells"] >= 1000)
)

In [10]:
adata.obs['sample.drawDate'] = pd.to_datetime(adata.obs['sample.drawDate'])
adata.obs['subject.birthYear'] = adata.obs['subject.birthYear'].astype(int)

# compute age
adata.obs['subject.age'] = (
    adata.obs['sample.drawDate'].dt.year - adata.obs['subject.birthYear']
)

/tmp/ipykernel_9522/2375413180.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs['sample.drawDate'] = pd.to_datetime(adata.obs['sample.drawDate'])


In [11]:
# Sample-level metadata
meta = adata.obs[
    [
        "sample.sampleKitGuid",
        "label.visitDetails",
        "sample.diseaseStatesRecordedAtVisit",
        "subject.biologicalSex",
        "sample.drawDate",
        "subject.birthYear",
        "subject.age",
        "subject.ethnicity",
        "subject.race",
        "subject.subjectGuid",
        "subject.cmv",
        "cohort.cohortGuid",
        "manual.time_stamp",
        "manual.category",
        "manual.flu_response"
    ]
].drop_duplicates()

In [12]:
# Merge counts into metadata
result = meta.merge(cell_counts, on="sample.sampleKitGuid", how="left")
result.to_csv("../../../data/rna/pseudobulk/outputs/bmmc_sample_kit_metadata.csv")

### 3. Filter gene set creation

In [13]:
%run ../../00-utilities/functions/python/filter_genes.py

#### Treatment timepoints

In [15]:
visit_pairs = [
    ("PreTx", "EI"),
    ("EI", "ASCT90d"),
    ("EI", "ASCT1y"),
    ("EI", "ASCT2y"),
    ("ASCT90d", "ASCT1y"),
    ("ASCT1y", "ASCT2y"),
]

for v1, v2 in visit_pairs:
    # keep only the two visits
    flt = adata.obs["label.visitDetails"].isin([v1, v2])
    adata_filtered = adata[flt].copy()

    out = f"../../../data/rna/pseudobulk/outputs/bmmc_l3_{v1}-vs-{v2}_filtered_gene_list.csv"

    # run filtering
    filtered_gene_df = get_filtered_genes_by_label(
        adata_filtered,
        celltype_col="aifi_plot_l3",
        min_frac=0.1,
        output_csv=out,
    )

    print(f"Saved {v1} vs {v2} at {out}")


Saved PreTx vs EI at outputs/bmmc_l3_PreTx-vs-EI_filtered_gene_list.csv
Saved EI vs ASCT90d at outputs/bmmc_l3_EI-vs-ASCT90d_filtered_gene_list.csv
Saved EI vs ASCT1y at outputs/bmmc_l3_EI-vs-ASCT1y_filtered_gene_list.csv
Saved EI vs ASCT2y at outputs/bmmc_l3_EI-vs-ASCT2y_filtered_gene_list.csv
Saved ASCT90d vs ASCT1y at outputs/bmmc_l3_ASCT90d-vs-ASCT1y_filtered_gene_list.csv
Saved ASCT1y vs ASCT2y at outputs/bmmc_l3_ASCT1y-vs-ASCT2y_filtered_gene_list.csv


#### BMMC Tx vs Healthy Comparisons

In [16]:
visits = [
    "PreTx",
    "EI",
    "ASCT90d",
    "ASCT1y",
    "ASCT2y"
]

for visit in visits:
    # keep healthy + current visit
    flt = adata.obs["label.visitDetails"].isin(["Healthy", visit])
    adata_filtered = adata[flt].copy()

    out = f"../../../data/rna/pseudobulk/outputs/bmmc_l3_healthy-vs-{visit}_filtered_gene_list.csv"

    # run filtering
    filtered_gene_df = get_filtered_genes_by_label(
        adata_filtered,
        celltype_col="aifi_plot_l3",
        min_frac=0.1,
        output_csv=out,
    )

    print(f"Saved Healthy vs {visit} at {out}")

Saved Healthy vs PreTx at outputs/bmmc_l3_healthy-vs-PreTx_filtered_gene_list.csv
Saved Healthy vs EI at outputs/bmmc_l3_healthy-vs-EI_filtered_gene_list.csv
Saved Healthy vs ASCT90d at outputs/bmmc_l3_healthy-vs-ASCT90d_filtered_gene_list.csv
Saved Healthy vs ASCT1y at outputs/bmmc_l3_healthy-vs-ASCT1y_filtered_gene_list.csv
Saved Healthy vs ASCT2y at outputs/bmmc_l3_healthy-vs-ASCT2y_filtered_gene_list.csv
